In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO

import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

def find_subdir(root, targets):
    targets = set([t.lower() for t in targets])
    best = None
    for dirpath, dirnames, filenames in os.walk(root):
        base = os.path.basename(dirpath).lower()
        if base in targets:
            img_files = [f for f in filenames if f.lower().endswith((".png",".jpg",".jpeg",".bmp"))]
            if len(img_files) > 0:
                best = dirpath
                break
    return best

root = path

images_dir = find_subdir(root, ["images", "imgs", "image", "rgb", "im"])
masks_dir  = find_subdir(root, ["masks", "mask", "labels", "label", "gt", "gts", "seg", "segmentation"])

if images_dir is None or masks_dir is None:
    raise FileNotFoundError(f"Could not find images/masks folders inside: {root}\nFound images_dir={images_dir}\nFound masks_dir={masks_dir}")

class SUIMDataset(Dataset):
    def __init__(self, images_dir, masks_dir, size=(256,256)):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.size = size
        self.images = sorted([f for f in os.listdir(images_dir) if f.lower().endswith((".png",".jpg",".jpeg",".bmp"))])
        self.masks  = sorted([f for f in os.listdir(masks_dir)  if f.lower().endswith((".png",".jpg",".jpeg",".bmp"))])

        if len(self.images) == 0 or len(self.masks) == 0:
            raise FileNotFoundError("Images or masks folder is empty.")
        if len(self.images) != len(self.masks):
            n = min(len(self.images), len(self.masks))
            self.images = self.images[:n]
            self.masks  = self.masks[:n]

        self.img_tf = transforms.Compose([
            transforms.Resize(self.size),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path  = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir,  self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        mask  = Image.open(mask_path)

        image = self.img_tf(image)
        mask  = mask.resize(self.size, resample=Image.NEAREST)
        mask  = torch.from_numpy(np.array(mask)).long()

        if mask.ndim == 3:
            mask = mask[..., 0]

        return image, mask

dataset = SUIMDataset(images_dir, masks_dir, size=(256,256))

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
g = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=g)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=0)

print("images_dir:", images_dir)
print("masks_dir :", masks_dir)
print("Total:", len(dataset), "| Train:", len(train_dataset), "| Val:", len(val_dataset))

images, masks = next(iter(train_loader))
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(images[0].permute(1,2,0).clamp(0,1))
plt.axis("off")
plt.subplot(1,2,2)
plt.imshow(masks[0].cpu().numpy())
plt.axis("off")
plt.show()

In [ ]:
# TO DO

!pip install -q segmentation-models-pytorch timm

import torch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
).to(device)

model

In [ ]:
# TO DO

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# TO DO

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 3

train_losses = []
val_losses = []

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = validate_one_epoch(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TO DO

import torch
import matplotlib.pyplot as plt

def show_predictions(model, loader, n=3):
    model.eval()
    images, masks = next(iter(loader))
    images = images[:n].to(device)
    masks = masks[:n].to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

    images = images.cpu()
    masks = masks.cpu()
    preds = preds.cpu()

    for i in range(n):
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plt.imshow(images[i].permute(1, 2, 0).clamp(0, 1))
        plt.title("Image")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(masks[i], vmin=0, vmax=7)
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(preds[i], vmin=0, vmax=7)
        plt.title("Prediction")
        plt.axis("off")

        plt.show()

show_predictions(model, val_loader, n=3)